# Day 2: EDA and Class Imbalance Analysis

This notebook explores the Kaggle Credit Card Fraud Detection dataset as part of Day 2 of the Credit Card Fraud Detection & Risk Scoring System project.

The main goal of this notebook is to understand the dataset and explore the fraud class imbalance, not to build any predictive models.

No modeling, preprocessing, or threshold tuning is performed on Day 2.

## Day 2 Objectives

- Load and validate the raw dataset
- Inspect dataset shape and columns
- Check missing values and duplicates
- Analyze fraud vs legitimate transaction distribution
- Explain why accuracy is misleading
- Generate and save EDA figures
- Prepare conclusions for Day 3

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Support kernels launched from either the repo root or notebooks/ directory.
candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root containing src/.")
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.data_loader import load_credit_card_data, summarize_dataset
from src.eda.imbalance_analysis import (
    generate_imbalance_summary,
    format_imbalance_summary_for_console,
    explain_accuracy_problem,
)
from src.eda.visualizations import generate_all_eda_figures

In [ ]:
DATA_PATH = PROJECT_ROOT / "data/raw/creditcard.csv"
FIGURES_DIR = PROJECT_ROOT / "reports/figures"

print(f"Using DATA_PATH: {DATA_PATH.resolve()}")
print(f"Using FIGURES_DIR: {FIGURES_DIR.resolve()}")

## 1. Load Dataset

The raw transaction data is loaded from `data/raw/creditcard.csv` and validated using the project's `load_credit_card_data` function, which checks that the expected schema (Time, V1-V28, Amount, Class) is present before returning the DataFrame.

In [ ]:
df = load_credit_card_data(DATA_PATH, validate=True)

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Dataset Overview

In [ ]:
summary = summarize_dataset(df)

print("Dataset Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

In [ ]:
df.info()

In [ ]:
df.describe().T.head(10)

## 3. Column Check

- `Time`: seconds elapsed between this transaction and the first transaction in the dataset.
- `V1` to `V28`: PCA-transformed anonymized features. They cannot be mapped back to real-world transaction attributes.
- `Amount`: the transaction amount.
- `Class`: the target variable, where 0 means legitimate and 1 means fraudulent.

In [ ]:
list(df.columns)

In [ ]:
df.dtypes.value_counts()

## 4. Missing Values and Duplicate Rows

In [ ]:
total_missing = df.isna().sum().sum()
print(f"Total missing values: {total_missing}")

missing_per_column = df.isna().sum()
missing_per_column = missing_per_column[missing_per_column > 0]

if missing_per_column.empty:
    print("No columns contain missing values.")
else:
    print("Columns with missing values:")
    print(missing_per_column)

duplicate_rows = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_rows}")

## 5. Class Distribution

- `Class = 0` means a legitimate transaction.
- `Class = 1` means a fraudulent transaction.

Fraud detection datasets like this one are typically highly imbalanced, with fraudulent transactions making up only a small fraction of the total.

In [ ]:
class_counts = df["Class"].value_counts().rename({0: "Legitimate", 1: "Fraud"})
class_counts_df = class_counts.rename("count").to_frame()
class_counts_df

In [ ]:
class_percentages = (df["Class"].value_counts(normalize=True) * 100).rename({0: "Legitimate", 1: "Fraud"})
class_percentages_df = class_percentages.rename("percentage").to_frame().round(4)
class_percentages_df

## 6. Class Imbalance Summary

In [ ]:
imbalance_summary = generate_imbalance_summary(df)
print(format_imbalance_summary_for_console(imbalance_summary))

## 7. Why Accuracy Is Misleading

Because legitimate transactions dominate this dataset, a naive model that always predicts "legitimate" can achieve very high accuracy while catching zero fraudulent transactions. This is why accuracy alone is not a trustworthy metric for this project.

In [ ]:
print(explain_accuracy_problem(df))

## 8. EDA Visualizations

This section generates and saves the following figures to `reports/figures/`:

- Class distribution plot
- Amount distribution by class
- Time distribution by class
- Correlation heatmap

In [ ]:
figure_paths = generate_all_eda_figures(df, output_dir=FIGURES_DIR)
figure_paths

### Generated Figures Preview

The cell below displays each generated figure inline. If a figure file is missing, a message is printed instead of raising an error.

In [ ]:
import matplotlib.image as mpimg

figure_files = {
    "Class Distribution": "class_distribution.png",
    "Amount Distribution by Class": "amount_distribution_by_class.png",
    "Time Distribution by Class": "time_distribution_by_class.png",
    "Correlation Heatmap": "correlation_heatmap.png",
}

for title, filename in figure_files.items():
    figure_path = FIGURES_DIR / filename
    if figure_path.exists():
        image = mpimg.imread(figure_path)
        plt.figure(figsize=(8, 6))
        plt.imshow(image)
        plt.axis("off")
        plt.title(title)
        plt.show()
    else:
        print(f"Figure not found: {figure_path}. Skipping display.")

## 9. Key Day 2 Findings

- The dataset contains anonymized PCA-transformed features `V1` to `V28`, which cannot be mapped back to real-world transaction attributes.
- The target variable (`Class`) is highly imbalanced, with legitimate transactions vastly outnumbering fraudulent ones.
- Fraudulent transactions are rare compared to legitimate transactions.
- Accuracy alone is not a suitable metric for evaluating fraud detection performance on this dataset.
- Future models must be evaluated using precision, recall, F1-score, PR-AUC, ROC-AUC, and the confusion matrix instead of relying on accuracy.

## 10. Day 3 Next Steps

- Build a leakage-safe preprocessing pipeline.
- Create a stratified train/validation/test split.
- Apply scaling where appropriate (e.g., to `Amount` and `Time`).
- Prepare the dataset and workflow for baseline model training.
- Keep fraud-focused evaluation metrics in focus throughout future model development.